# Feature Engineering

Customer-level feature engineering for the retention prediction model (Zain's contribution). This starts from the cleaned transaction data produced in the data-cleaning step and builds the RFM + behavioral feature set used to train the model.

In [ ]:
import pandas as pd

In [ ]:
# Load the cleaned transactions produced by the data-cleaning step
df_clean = pd.read_csv("../Data/clean_transactions.csv", parse_dates=["InvoiceDate"])
df_clean.shape

## Define the feature and target time windows

We need 30 days of future data to determine whether a customer purchased again, so the cutoff date is moved 30 days back from the last transaction.

In [ ]:
cutoff_date = df_clean["InvoiceDate"].max() - pd.Timedelta(days=30)
cutoff_date

In [ ]:
# Separating the clean data into the feature period and the target period
feature_data = df_clean[df_clean["InvoiceDate"] <= cutoff_date]
target_data = df_clean[df_clean["InvoiceDate"] > cutoff_date]

feature_data.shape, target_data.shape

**Feature Engineering**

Since the dataset was transactional, but our model needs customer-level information.

In [ ]:
# Feature 1: Recency
last_purchase = feature_data.groupby("CustomerID")["InvoiceDate"].max()
recency = (cutoff_date - last_purchase).dt.days
recency.head()

In [ ]:
# Feature 2: Frequency
frequency = feature_data.groupby("CustomerID")["InvoiceNo"].nunique()
frequency.head()

In [ ]:
# Feature 3: Monetary
feature_data["TotalAmount"] = feature_data["Quantity"] * feature_data["UnitPrice"]

# Calculate Monetary per customer
monetary = feature_data.groupby("CustomerID")["TotalAmount"].sum()
monetary.head()

In [ ]:
rfm = pd.concat([recency, frequency, monetary], axis=1)
rfm.columns = ["Recency", "Frequency", "Monetary"]
rfm.head()

In [ ]:
# Feature 4: Total Quantity
total_quantity = feature_data.groupby("CustomerID")["Quantity"].sum()
total_quantity.head()

In [ ]:
# Feature 5: Unique Products
unique_products = feature_data.groupby("CustomerID")["StockCode"].nunique()
rfm["UniqueProducts"] = unique_products
rfm.head()

In [ ]:
# Feature 6: Average Order Value
rfm["AverageOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm.head()

In [ ]:
# Feature 7: Purchase Days
feature_data["PurchaseDate"] = feature_data["InvoiceDate"].dt.date
unique_purchase_days = feature_data.groupby("CustomerID")["PurchaseDate"].nunique()
rfm["UniquePurchaseDays"] = unique_purchase_days
rfm.head()

In [ ]:
# Feature 8: Customer Lifespan
first_purchase = feature_data.groupby("CustomerID")["InvoiceDate"].min()
customer_lifespan = (last_purchase - first_purchase).dt.days
rfm["CustomerLifespan"] = customer_lifespan
rfm.head()

In [ ]:
# Feature 9: Orders in the last 30 days of the feature window
last_30_days_start = cutoff_date - pd.Timedelta(days=30)

orders_last_30_days = (
    feature_data[feature_data["InvoiceDate"] > last_30_days_start]
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

rfm["OrdersLast30Days"] = orders_last_30_days
rfm["OrdersLast30Days"] = rfm["OrdersLast30Days"].fillna(0).astype(int)
rfm.head()

In [ ]:
# Target: did the customer purchase again in the 30-day target window?
repeat_customers = target_data["CustomerID"].unique()
rfm["Target"] = rfm.index.isin(repeat_customers).astype(int)
rfm["Target"].value_counts()

## Validate the engineered feature set

In [ ]:
rfm.shape

In [ ]:
rfm.isnull().sum()

In [ ]:
rfm.dtypes

In [ ]:
rfm.describe().T

## Save engineered features

Handed off as a customer-level feature table for the modeling step.

In [ ]:
rfm.to_csv("../Data/rfm_features.csv")